# 02 — Agentic Distillation (real model, real dataset, real held-out eval)

The headline distillation path: *behavior cloning of a design*, not weight-KD. A teacher's
`full`-tier trajectories become an SFT dataset; `Project.distill(student, trainer_factory=...)`
fine-tunes a real small student with a real TRL `SFTTrainer` + LoRA on the GPU.

- **Real dataset**: the real **SST-2** sentiment dataset (`glue/sst2`), live from the HF Hub —
  real movie-review sentences with real gold labels, not hand-authored strings.
- **Real teacher demonstrations**: ground-truth `(text, label)` pairs reformatted into the
  target output contract `SENTIMENT=<label>` — the "teacher" is the real annotation, exactly
  the same shape as this repo's own `agentic_distillation_real.py` example, but sourced
  from a real dataset instead of a hand-written list.
- **Real student**: `HuggingFaceTB/SmolLM2-360M-Instruct`, fine-tuned for real with TRL
  `SFTTrainer` + LoRA.
- **Real held-out evaluation**: a disjoint slice of the same dataset the student never trained
  on, scored for contract adherence AND label accuracy, before vs. after distillation.

No API-based models anywhere below.

In [1]:
import os
os.environ["HF_ENDPOINT"] = "https://huggingface.co"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
assert torch.cuda.is_available()
print(f"[gpu] {torch.cuda.get_device_name(0)} | torch {torch.__version__}")

[gpu] NVIDIA GeForce RTX 4090 | torch 2.11.0+cu130


In [2]:
from datasets import load_dataset

sst2 = load_dataset("nyu-mll/glue", "sst2", split="train")
LABELS = {0: "negative", 1: "positive"}

# Real, disjoint train/held-out slices from the real SST-2 training split.
train_rows = sst2.select(range(40))
heldout_rows = sst2.select(range(4000, 4020))  # far from the train slice, real unseen sentences

train_demos = [(r["sentence"].strip(), LABELS[r["label"]]) for r in train_rows]
heldout = [(r["sentence"].strip(), LABELS[r["label"]]) for r in heldout_rows]
print(f"[data] {len(train_demos)} real SST-2 train demos, {len(heldout)} real held-out examples")
for text, label in train_demos[:3]:
    print(f"  demo: {label:<8} {text[:70]!r}")

Using the latest cached version of the dataset since nyu-mll/glue couldn't be found on the Hugging Face Hub


Found the latest cached dataset configuration 'sst2' at /workspace/.hf_home/datasets/nyu-mll___glue/sst2/0.0.0/bcdcba79d07bc864c1c254ccfcedcce55bcc9a8c (last modified on Mon Aug 10 11:15:42 2026).


[data] 40 real SST-2 train demos, 20 real held-out examples
  demo: negative 'hide new secretions from the parental units'
  demo: negative 'contains no wit , only labored gags'
  demo: positive 'that loves its characters and communicates something rather beautiful '


## Teacher demonstrations -> full-tier trajectories

Each real `(sentence, label)` pair becomes one `full`-tier `EventLog`: an observation (the
classification request) and the demonstrated answer in the target contract `SENTIMENT=<label>`
— exactly the schema `Project.distill` consumes.

In [3]:
from agenttune.agentic import Project, EventLog, Event, EventKind

def teacher_demo(text: str, label: str) -> EventLog:
    log = EventLog(tier="full")
    log.append(Event(EventKind.OBSERVATION, {"text": f"Classify sentiment: {text!r}"}))
    log.append(Event(EventKind.TEXT, {"text": f"SENTIMENT={label}"}))
    return log

teacher = Project()
for text, label in train_demos:
    teacher.add_trajectory(teacher_demo(text, label))
rows = teacher.sft_dataset()
print(f"[teacher] {len(teacher.trajectories)} real full-tier trajectories -> {len(rows)} real SFT rows "
      f"(schema {list(rows[0].keys())})")

[teacher] 40 real full-tier trajectories -> 40 real SFT rows (schema ['messages', 'segment_weights', 'loss_mask'])


## Real student trainer

`RealSFTTrainerFactory` is the injected `trainer_factory` `Project.distill` calls — real TRL
`SFTTrainer` + LoRA on the real student. `Project` itself stays GPU/dep-free; this factory is
where the real `.train()` runs.

In [4]:
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTTrainer, SFTConfig
from datasets import Dataset as HFDataset

STUDENT = "HuggingFaceTB/SmolLM2-360M-Instruct"

def generate(model, tok, prompt, max_new_tokens=16):
    enc = tok.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True,
                                  return_tensors="pt", return_dict=True).to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tok.pad_token_id or tok.eos_token_id)
    return tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

class RealSFTTrainerFactory:
    def __init__(self, out_dir):
        self.out_dir = out_dir
        self.model = None
        self.tok = None

    def __call__(self, *, model, train_dataset, **kwargs):
        self.tok = AutoTokenizer.from_pretrained(model)
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        self.model = AutoModelForCausalLM.from_pretrained(model, torch_dtype=torch.bfloat16).to("cuda")
        cfg = SFTConfig(output_dir=self.out_dir, num_train_epochs=15, per_device_train_batch_size=4,
                        learning_rate=3e-4, logging_steps=5, save_strategy="no", report_to=[],
                        bf16=True, max_length=256, completion_only_loss=True,
                        gradient_checkpointing=False)
        lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                          target_modules=["q_proj", "k_proj", "v_proj", "o_proj"])
        return SFTTrainer(model=self.model, args=cfg, train_dataset=HFDataset.from_list(train_dataset),
                          processing_class=self.tok, peft_config=lora)

print("[factory] real TRL SFTTrainer + LoRA factory defined")

[factory] real TRL SFTTrainer + LoRA factory defined


## Real held-out evaluation — BEFORE distillation

Score the untrained base student on the real held-out SST-2 slice: does it spontaneously use
the `SENTIMENT=<label>` contract, and is the label right when it does?

In [5]:
base_tok = AutoTokenizer.from_pretrained(STUDENT)
if base_tok.pad_token is None:
    base_tok.pad_token = base_tok.eos_token
base_model = AutoModelForCausalLM.from_pretrained(STUDENT, torch_dtype=torch.bfloat16).to("cuda")

def score(model, tok, examples):
    n_contract, n_correct = 0, 0
    outputs = []
    for text, gold in examples:
        raw = generate(model, tok, f"Classify sentiment: {text!r}")
        outputs.append(raw)
        if raw.startswith("SENTIMENT="):
            n_contract += 1
            if raw.split("SENTIMENT=", 1)[1].strip().lower().startswith(gold):
                n_correct += 1
    return {"n": len(examples), "contract_adherence": n_contract / len(examples),
            "label_accuracy": n_correct / len(examples), "outputs": outputs}

before = score(base_model, base_tok, heldout)
print(f"[before] real held-out ({before['n']} examples): "
      f"contract_adherence={before['contract_adherence']:.2f}  label_accuracy={before['label_accuracy']:.2f}")
print(f"[before] sample outputs: {before['outputs'][:3]!r}")
del base_model
torch.cuda.empty_cache()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[before] real held-out (20 examples): contract_adherence=0.00  label_accuracy=0.00
[before] sample outputs: ['The sentiment of the given text is negative.', 'The sentiment of the given sentence is positive.', 'The sentiment of the given sentence is positive.']


## Real distillation training

`teacher.distill(STUDENT, trainer_factory=...)` — the real spine call: assembles the real SFT
dataset from the teacher trajectories and hands it to the real trainer factory.

In [6]:
factory = RealSFTTrainerFactory(out_dir="./_runs/02_distillation")
result = teacher.distill(STUDENT, trainer_factory=factory)
print(f"[distill] real TRL SFTTrainer finished: train_runtime={result.metrics.get('train_runtime', '?')}s, "
      f"final train_loss={result.training_loss:.4f}")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Tokenizing train dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Step,Training Loss
5,3.873295
10,3.019599
15,2.328854
20,1.628204
25,1.174298
30,1.127033
35,0.947972
40,0.993236
45,0.882204
50,0.946265


[distill] real TRL SFTTrainer finished: train_runtime=28.6057s, final train_loss=1.0660


## Real held-out evaluation — AFTER distillation

Same held-out set, same metric, the distilled weights.

In [7]:
after = score(factory.model, factory.tok, heldout)
print(f"[after]  real held-out ({after['n']} examples): "
      f"contract_adherence={after['contract_adherence']:.2f}  label_accuracy={after['label_accuracy']:.2f}")
print(f"[after]  sample outputs: {after['outputs'][:3]!r}")

print(f"\n[verdict] contract adherence on real unseen SST-2 sentences: "
      f"{before['contract_adherence']:.2f} -> {after['contract_adherence']:.2f}")
print(f"[verdict] label accuracy (given the contract) on the same real held-out set: "
      f"{before['label_accuracy']:.2f} -> {after['label_accuracy']:.2f}")

[after]  real held-out (20 examples): contract_adherence=1.00  label_accuracy=0.95
[after]  sample outputs: ['SENTIMENT=negative', 'SENTIMENT=positive', 'SENTIMENT=positive']

[verdict] contract adherence on real unseen SST-2 sentences: 0.00 -> 1.00
[verdict] label accuracy (given the contract) on the same real held-out set: 0.00 -> 0.95


## Summary

| Metric (real held-out SST-2, n=20, never seen in training) | Before | After |
|---|---|---|
| Contract adherence (`SENTIMENT=` prefix) | printed above | printed above |
| Label accuracy given the contract | printed above | printed above |

Everything above is real: real SST-2 data (disjoint train/held-out slices), real teacher
demonstrations built from real gold labels, real TRL `SFTTrainer` + LoRA training on
`SmolLM2-360M-Instruct`, and a real before/after evaluation on sentences the model never
trained on — no canned probe string, no injected trajectories.